In [1]:
# ============================================================
# TASK 1: Character-Level Name Generation
# Vanilla RNN | BiLSTM Encoder-Decoder | RNN + Attention
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import random
from google.colab import files

# ============================================================
# STEP 0: Upload Dataset
# ============================================================

print("Upload your TrainingNames.txt file...")
uploaded = files.upload()

with open("TrainingNames.txt", "r", encoding="utf-8") as f:
    names = [line.strip().lower() for line in f if line.strip()]

print(f"Total names loaded: {len(names)}")
print(f"Sample names: {names[:5]}")

# ============================================================
# STEP 1: Build Vocabulary
# ============================================================

all_chars      = sorted(set("".join(names)))
special_tokens = ["<PAD>", "<SOS>", "<EOS>"]
vocab          = special_tokens + all_chars
char2idx       = {ch: idx for idx, ch in enumerate(vocab)}
idx2char       = {idx: ch  for idx, ch in enumerate(vocab)}

PAD_IDX    = char2idx["<PAD>"]
SOS_IDX    = char2idx["<SOS>"]
EOS_IDX    = char2idx["<EOS>"]
VOCAB_SIZE = len(vocab)

print(f"Vocabulary size : {VOCAB_SIZE}")
print(f"Characters      : {all_chars}")

# ============================================================
# STEP 2: Dataset Preparation
# ============================================================

def name_to_tensor(name):
    """
    input  : [<SOS>, c1, c2, ..., cn]
    target : [c1, c2, ..., cn, <EOS>]
    """
    indices       = [char2idx[ch] for ch in name]
    input_tensor  = torch.tensor([SOS_IDX] + indices, dtype=torch.long)
    target_tensor = torch.tensor(indices + [EOS_IDX],  dtype=torch.long)
    return input_tensor, target_tensor

# ============================================================
# STEP 3: ARCHITECTURE 1 — Vanilla RNN
# ============================================================

class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64,
                 hidden_size=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.embedding   = nn.Embedding(vocab_size, embedding_dim,
                                        padding_idx=PAD_IDX)
        self.rnn         = nn.RNN(embedding_dim, hidden_size, num_layers,
                                  batch_first=True, nonlinearity='tanh',
                                  dropout=dropout if num_layers > 1 else 0)
        self.dropout     = nn.Dropout(dropout)
        self.fc          = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        out, hidden = self.rnn(self.dropout(self.embedding(x)), hidden)
        return self.fc(self.dropout(out)), hidden

    def init_hidden(self, batch_size, device):
        return torch.zeros(self.num_layers, batch_size,
                           self.hidden_size, device=device)


# ============================================================
# STEP 4: ARCHITECTURE 2 — BiLSTM Encoder-Decoder
# ============================================================

class BiLSTMEncoderDecoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64,
                 hidden_size=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        self.embedding = nn.Embedding(vocab_size, embedding_dim,
                                      padding_idx=PAD_IDX)
        self.dropout   = nn.Dropout(dropout)

        # Encoder — bidirectional LSTM reads the seed token
        self.encoder = nn.LSTM(
            embedding_dim, hidden_size, num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        # Project fwd+bwd hidden/cell states (2H) down to H
        # so decoder always works with hidden_size-dim states
        self.h_proj = nn.Linear(hidden_size * 2, hidden_size)
        self.c_proj = nn.Linear(hidden_size * 2, hidden_size)

        # Decoder — standard unidirectional LSTM for generation
        self.decoder = nn.LSTM(
            embedding_dim, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.fc = nn.Linear(hidden_size, vocab_size)

    def _encode_sos(self, device):
        """
        Always encode just [SOS] — identical at train and inference.
        Returns (h, c) shaped (num_layers, batch=1, hidden_size).
        """
        sos    = torch.tensor([[SOS_IDX]], dtype=torch.long, device=device)
        emb    = self.dropout(self.embedding(sos))
        _, (h, c) = self.encoder(emb)

        B = 1
        # BiLSTM stacks layers as [fwd_L0, bwd_L0, fwd_L1, bwd_L1 ...]
        # Reshape to (num_layers, 2, B, H) then concat fwd+bwd
        h = h.view(self.num_layers, 2, B, self.hidden_size)
        c = c.view(self.num_layers, 2, B, self.hidden_size)
        h = torch.tanh(self.h_proj(
                torch.cat([h[:, 0, :, :], h[:, 1, :, :]], dim=2)))
        c = torch.tanh(self.c_proj(
                torch.cat([c[:, 0, :, :], c[:, 1, :, :]], dim=2)))
        # h, c: (num_layers, 1, hidden_size)
        return h, c

    def forward(self, x, hidden=None):
        """
        Training forward:
          1. Encode [SOS] to get decoder initial state (same as inference)
          2. Decode the full input sequence with teacher forcing
        This ensures zero train/inference mismatch.
        """
        device = x.device
        B      = x.size(0)

        # Step 1: encode SOS to get initial decoder state
        h, c   = self._encode_sos(device)

        # Expand to match batch size > 1 if needed
        h = h.expand(-1, B, -1).contiguous()
        c = c.expand(-1, B, -1).contiguous()

        # Step 2: teacher-force decode the full sequence
        emb         = self.dropout(self.embedding(x))
        out, hidden = self.decoder(emb, (h, c))
        logits      = self.fc(self.dropout(out))
        return logits, hidden

    def init_hidden(self, batch_size, device):
        h = torch.zeros(self.num_layers, batch_size,
                        self.hidden_size, device=device)
        c = torch.zeros(self.num_layers, batch_size,
                        self.hidden_size, device=device)
        return (h, c)


# ============================================================
# STEP 5: ARCHITECTURE 3 — RNN with Attention
# ============================================================

class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v    = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: (B, H), encoder_outputs: (B, T, H)
        T = encoder_outputs.size(1)
        h = hidden.unsqueeze(1).expand(-1, T, -1)
        e = torch.tanh(self.attn(
                torch.cat([h, encoder_outputs], dim=2)))
        w   = torch.softmax(self.v(e).squeeze(2), dim=1)
        ctx = torch.bmm(w.unsqueeze(1), encoder_outputs).squeeze(1)
        return ctx, w


class RNNWithAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64,
                 hidden_size=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.embedding   = nn.Embedding(vocab_size, embedding_dim,
                                        padding_idx=PAD_IDX)
        self.rnn         = nn.RNN(embedding_dim, hidden_size, num_layers,
                                  batch_first=True, nonlinearity='tanh',
                                  dropout=dropout if num_layers > 1 else 0)
        self.attention   = Attention(hidden_size)
        self.dropout     = nn.Dropout(dropout)
        self.fc          = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x, hidden=None):
        """
        Causal training forward:
        At each step t, attend ONLY over rnn_out[:, :t+1, :]
        This exactly mirrors the decode_step inference loop.
        """
        emb             = self.dropout(self.embedding(x))
        rnn_out, hidden = self.rnn(emb, hidden)
        T      = rnn_out.size(1)
        logits = []
        for t in range(T):
            # Attend only over past + current outputs (causal)
            ctx, _ = self.attention(hidden[-1], rnn_out[:, :t+1, :])
            step   = torch.cat([rnn_out[:, t, :], ctx], dim=1)
            logits.append(self.fc(self.dropout(step)))
        return torch.stack(logits, dim=1), hidden

    def init_hidden(self, batch_size, device):
        return torch.zeros(self.num_layers, batch_size,
                           self.hidden_size, device=device)

    def decode_step(self, x, hidden, history):
        """
        Single inference step — grows history by 1 each call.
        x      : (1, 1) current character index
        hidden : current RNN hidden state
        history: (1, t, H) all previous RNN outputs
        Returns logits, new hidden, updated history
        """
        emb             = self.dropout(self.embedding(x))
        rnn_out, hidden = self.rnn(emb, hidden)
        # Append new RNN output to running history
        history         = torch.cat([history, rnn_out], dim=1)
        ctx, _          = self.attention(hidden[-1], history)
        combined        = torch.cat([rnn_out.squeeze(1), ctx], dim=1)
        logits          = self.fc(self.dropout(combined))
        return logits, hidden, history


# ============================================================
# STEP 6: Count Trainable Parameters
# ============================================================

def count_parameters(model):
    return sum(p.numel() for p in model.parameters()
               if p.requires_grad)

# ============================================================
# STEP 7: Training Loop
# ============================================================

def train_model(model, names, epochs=100, lr=0.003,
                print_every=10, device='cpu',
                label_smoothing=0.1, weight_decay=1e-5):
    model.to(device)

    # Label smoothing prevents overconfident predictions
    # which cause repetitive generation
    criterion = nn.CrossEntropyLoss(
        ignore_index=PAD_IDX,
        label_smoothing=label_smoothing
    )

    optimizer = optim.Adam(model.parameters(), lr=lr,
                           weight_decay=weight_decay)

    # CosineAnnealingWarmRestarts gives the model periodic
    # chances to escape local minima — helps RNN + Attention
    # converge to lower loss than plain cosine decay
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=25, T_mult=2
    )

    loss_history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        random.shuffle(names)

        for name in names:
            inp, tgt = name_to_tensor(name)
            inp = inp.unsqueeze(0).to(device)
            tgt = tgt.unsqueeze(0).to(device)

            optimizer.zero_grad()

            if isinstance(model, BiLSTMEncoderDecoder):
                # BiLSTM encodes SOS internally — no hidden needed
                logits, _ = model(inp)
            else:
                hidden = model.init_hidden(1, device)
                logits, _ = model(inp, hidden)

            loss = criterion(
                logits.view(-1, VOCAB_SIZE),
                tgt.view(-1)
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=5.0)
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        avg = total_loss / len(names)
        loss_history.append(avg)

        if epoch % print_every == 0:
            print(f"  Epoch {epoch:>3}/{epochs}  |  "
                  f"Avg Loss: {avg:.4f}  |  "
                  f"LR: {scheduler.get_last_lr()[0]:.6f}")

    return loss_history


# ============================================================
# STEP 8: Generation Functions
# ============================================================

def generate_name_rnn(model, device, max_len=15, temperature=0.8):
    """Standard autoregressive generation for Vanilla RNN."""
    model.eval()
    with torch.no_grad():
        inp    = torch.tensor([[SOS_IDX]], dtype=torch.long,
                              device=device)
        hidden = model.init_hidden(1, device)
        chars  = []

        for _ in range(max_len):
            logits, hidden = model(inp, hidden)
            probs    = torch.softmax(logits[0, 0] / temperature, dim=0)
            next_idx = torch.multinomial(probs, 1).item()

            if next_idx == EOS_IDX:
                break
            if next_idx not in (PAD_IDX, SOS_IDX):
                chars.append(idx2char[next_idx])
            inp = torch.tensor([[next_idx]], dtype=torch.long,
                               device=device)

    name = "".join(chars)
    return name.capitalize() if len(name) > 1 else ""


def generate_name_bilstm(model, device, max_len=15, temperature=0.8):
    """
    BiLSTM generation — encode SOS then autoregressively decode.
    Fully consistent with the fixed training forward() above.
    """
    model.eval()
    with torch.no_grad():
        # Get warm decoder state from encoding SOS
        h, c  = model._encode_sos(device)
        inp   = torch.tensor([[SOS_IDX]], dtype=torch.long,
                             device=device)
        chars = []

        for _ in range(max_len):
            emb         = model.dropout(model.embedding(inp))
            out, (h, c) = model.decoder(emb, (h, c))

            # out: (1, 1, hidden_size) → squeeze to (hidden_size,)
            logits   = model.fc(out[0, 0])
            probs    = torch.softmax(logits / temperature, dim=0)
            next_idx = torch.multinomial(probs, 1).item()

            if next_idx == EOS_IDX:
                break
            if next_idx not in (PAD_IDX, SOS_IDX):
                chars.append(idx2char[next_idx])
            inp = torch.tensor([[next_idx]], dtype=torch.long,
                               device=device)

    name = "".join(chars)
    return name.capitalize() if len(name) > 1 else ""


def generate_name_attn(model, device, max_len=15, temperature=0.8):
    """
    RNN+Attention generation.
    Key fix: bootstrap history with the real SOS RNN output,
    then grow it one step at a time using decode_step.
    This exactly mirrors the causal training loop.
    """
    model.eval()
    with torch.no_grad():
        inp    = torch.tensor([[SOS_IDX]], dtype=torch.long,
                              device=device)
        hidden = model.init_hidden(1, device)

        # Bootstrap: run SOS through RNN to seed the history
        # with one real vector instead of zeros
        emb             = model.dropout(model.embedding(inp))
        rnn_out, hidden = model.rnn(emb, hidden)
        history         = rnn_out   # (1, 1, H) — first real history entry

        # Predict the first character from SOS context
        ctx, _   = model.attention(hidden[-1], history)
        combined = torch.cat([rnn_out.squeeze(1), ctx], dim=1)
        logits   = model.fc(model.dropout(combined))
        probs    = torch.softmax(logits[0] / temperature, dim=0)
        next_idx = torch.multinomial(probs, 1).item()

        chars = []
        if next_idx == EOS_IDX:
            return ""
        if next_idx not in (PAD_IDX, SOS_IDX):
            chars.append(idx2char[next_idx])

        inp = torch.tensor([[next_idx]], dtype=torch.long,
                           device=device)

        # Continue generating using decode_step (history grows each step)
        for _ in range(max_len - 1):
            logits, hidden, history = model.decode_step(
                inp, hidden, history)
            probs    = torch.softmax(logits[0] / temperature, dim=0)
            next_idx = torch.multinomial(probs, 1).item()

            if next_idx == EOS_IDX:
                break
            if next_idx not in (PAD_IDX, SOS_IDX):
                chars.append(idx2char[next_idx])
            inp = torch.tensor([[next_idx]], dtype=torch.long,
                               device=device)

    name = "".join(chars)
    return name.capitalize() if len(name) > 1 else ""


def generate_batch(gen_fn, model, device, n=10, temperature=0.8):
    """Generate n valid names (>2 chars) from any model."""
    results, attempts = [], 0
    while len(results) < n and attempts < n * 30:
        name = gen_fn(model, device, temperature=temperature)
        if len(name) > 2:
            results.append(name)
        attempts += 1
    return results if results else ["(model needs more training)"]


# ============================================================
# STEP 9: Run Everything
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

EMBEDDING_DIM = 64
HIDDEN_SIZE   = 256
NUM_LAYERS    = 2
DROPOUT       = 0.3
EPOCHS        = 100
LR_RNN        = 0.003
LR_LSTM       = 0.001
LR_ATTN       = 0.005

print("\n" + "="*60)
print("         HYPERPARAMETER CONFIGURATION")
print("="*60)
print(f"  Embedding dim  : {EMBEDDING_DIM}")
print(f"  Hidden size    : {HIDDEN_SIZE}")
print(f"  Num layers     : {NUM_LAYERS}")
print(f"  Dropout        : {DROPOUT}")
print(f"  Epochs         : {EPOCHS}")
print(f"  LR (RNN)       : {LR_RNN}")
print(f"  LR (BiLSTM)    : {LR_LSTM}")
print(f"  LR (Attention) : {LR_ATTN}")
print("="*60)

rnn_model  = VanillaRNN(VOCAB_SIZE, EMBEDDING_DIM,
                        HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
lstm_model = BiLSTMEncoderDecoder(VOCAB_SIZE, EMBEDDING_DIM,
                                  HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
attn_model = RNNWithAttention(VOCAB_SIZE, EMBEDDING_DIM,
                              HIDDEN_SIZE, NUM_LAYERS, DROPOUT)

print("\n" + "="*60)
print("         TRAINABLE PARAMETER COUNT")
print("="*60)
print(f"  Vanilla RNN             : {count_parameters(rnn_model):>10,}")
print(f"  BiLSTM Encoder-Decoder  : {count_parameters(lstm_model):>10,}")
print(f"  RNN + Attention         : {count_parameters(attn_model):>10,}")
print("="*60)

print("\n[1/3] Training Vanilla RNN...")
rnn_losses = train_model(
    rnn_model, names, epochs=EPOCHS,
    lr=LR_RNN, print_every=10, device=device,
    label_smoothing=0.1, weight_decay=1e-5
)

print("\n[2/3] Training BiLSTM Encoder-Decoder...")
lstm_losses = train_model(
    lstm_model, names, epochs=EPOCHS,
    lr=LR_LSTM, print_every=10, device=device,
    label_smoothing=0.1, weight_decay=1e-4
)

print("\n[3/3] Training RNN + Attention...")
attn_losses = train_model(
    attn_model, names, epochs=EPOCHS,
    lr=LR_ATTN, print_every=10, device=device,
    label_smoothing=0.1, weight_decay=1e-5
)

print("\n" + "="*60)
print("    GENERATED NAMES (temperature=0.8, 10 names each)")
print("="*60)

configs = [
    ("Vanilla RNN",            rnn_model,  generate_name_rnn),
    ("BiLSTM Encoder-Decoder", lstm_model, generate_name_bilstm),
    ("RNN + Attention",        attn_model, generate_name_attn),
]

for label, model, gen_fn in configs:
    samples = generate_batch(gen_fn, model, device,
                             n=10, temperature=0.8)
    print(f"\n  {label}:")
    for name in samples:
        print(f"    {name}")

print("\n" + "="*60)
print("Training complete.")
print("="*60)

# Save weights for Task 2 and Task 3
torch.save(rnn_model.state_dict(),  "vanilla_rnn.pt")
torch.save(lstm_model.state_dict(), "blstm.pt")
torch.save(attn_model.state_dict(), "rnn_attention.pt")
print("\nModel weights saved.")

files.download("vanilla_rnn.pt")
files.download("blstm.pt")
files.download("rnn_attention.pt")

Upload your TrainingNames.txt file...


Saving TrainingNames.txt to TrainingNames.txt
Total names loaded: 1000
Sample names: ['aarav', 'aditya', 'akash', 'arjun', 'arnav']
Vocabulary size : 29
Characters      : ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

Using device: cuda

         HYPERPARAMETER CONFIGURATION
  Embedding dim  : 64
  Hidden size    : 256
  Num layers     : 2
  Dropout        : 0.3
  Epochs         : 100
  LR (RNN)       : 0.003
  LR (BiLSTM)    : 0.001
  LR (Attention) : 0.005

         TRAINABLE PARAMETER COUNT
  Vanilla RNN             :    223,325
  BiLSTM Encoder-Decoder  :  3,364,445
  RNN + Attention         :    362,333

[1/3] Training Vanilla RNN...
  Epoch  10/100  |  Avg Loss: 2.5408  |  LR: 0.001964
  Epoch  20/100  |  Avg Loss: 2.2926  |  LR: 0.000286
  Epoch  30/100  |  Avg Loss: 2.6361  |  LR: 0.002927
  Epoch  40/100  |  Avg Loss: 2.5757  |  LR: 0.002382
  Epoch  50/100  |  Avg Loss: 2.4570  |  LR: 0.00150

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>